# Day I - C) Error estimation and mesh refinement

Using a simple 2D inductor computation, we will illustrate the influence of the mesh on the accuracy of the solution, as well as classical error estimators used to drive adaptive mesh refinement.

- **<font color='green'>[RUN & OBSERVE]</font>** $\rightarrow$ the cell should be run directly without modification
- **<font color='orange'>[RUN & PLAY]</font>** $\rightarrow$ the cell can be run directly, but some parameters should be changed interactively
- **<font color='red'>[FILL & RUN]</font>**    $\rightarrow$ the cell should be filled before being run
- **<font color='magenta'>[FILL & PLAY]</font>** $\rightarrow$ the cell should be filled, and then some parameters should be changed interactively.

## 1) Magnetostatic computation

In [ ]:
# configuration of physical parameters

from ngsolve import pi
mu0 = 4e-7 * pi  # H/m, void magnetic permeability
mu_r = 1000      # relative permeability
J = 10e6         # source current density A/m^2

In [ ]:
from ngsolve import H1, CF, GridFunction, grad, dx, LinearForm, BilinearForm
from time import time

def curl(a):
    return CF(((0,1),(-1,0)), dims = (2,2)) * grad(a)

def solve_magnetostatic(mesh):
    """ Solve linear magnetostatic problem """
    # finite element space
    t0 = time()
    fes = H1(mesh, order = 1, dirichlet = "out")
    
     # variational formulation
    a, v = fes.TnT()
    A = BilinearForm(fes)
    A += 1/mu0 * curl(a) * curl(v) * dx("air|condP|condN")
    A += 1/(1000*mu0) * curl(a) * curl(v) * dx("iron")
    l = LinearForm(v * J * dx("condP") - v * J * dx("condN"))

    # Assemble matrix and solver 
    K = A.Assemble().mat
    f = l.Assemble().vec
    sol = GridFunction(fes)
    sol.vec.data = K.Inverse(freedofs = fes.FreeDofs()) * f
    print(f"Solution computed in {time()-t0 :.3f} s ({mesh.nv} DoFs)")

    return sol

In [ ]:
# Small test
from utils.myGeometries import gapedInductorrefine
from ngsolve.webgui import Draw

mesh = gapedInductorrefine(airgap = 0, h = 0.001)
sol = solve_magnetostatic(mesh)
Draw(sol)

In [ ]:
# Run with different meshes
hList = [0.005, 0.0025, 0.001, 0.0005, 0.00025]
solList = [solve_magnetostatic(gapedInductorrefine(airgap = 0, h = h))for h in hList]

In [ ]:
# Plot the results

index = -1
print(f"max mesh size = {hList[index]}")
Draw(solList[index])

## 2) Global magnetic energy
reload the soluiton for the global magnetic energy computation, no need to run again the solver..just post-processing

In [ ]:
from ngsolve import CF, Integrate


def magnetic_energy_density(a):
    B = curl(a)
    mesh = a.space.mesh
    return mesh.MaterialCF({"iron": B**2 /(2 * mu0 * mu_r) }, default = B**2 /(2 * mu0))
    
def magnetic_energy(a):
    return Integrate( magnetic_energy_density(a),mesh)

def total_energy_density(a):
    B = curl(a)
    mesh = a.space.mesh
    return mesh.MaterialCF({"iron":  B**2 /(2 * mu0 * mu_r),
                            "condP": B**2 /(2 * mu0) - a*J,
                            "condN": B**2 /(2 * mu0) + a*J},
                           default = B**2 /(2 * mu0))

def total_energy(a):
    return Integrate( total_energy_density(a), mesh)

# Compute the magnetostatic energy stored
energyList = [magnetic_energy(sol) for sol in solList] 
totalEnergyList = [total_energy(sol) for sol in solList] 

In [ ]:
# Display

import matplotlib.pyplot as plt

fig, ax1 = plt.subplots()

color = 'tab:red'
ax1.set_xlabel('h')
ax1.set_ylabel('Magnetic Energy (J)', color=color)
ax1.semilogx(hList, energyList, ':o', color=color)
ax1.tick_params(axis='y', labelcolor=color)

ax2 = ax1.twinx()  # instantiate a second Axes that shares the same x-axis

color = 'tab:blue'
ax2.set_ylabel('Total Energy (J)', color=color)
ax2.semilogx(hList, totalEnergyList, ':s')
ax2.tick_params(axis='y', labelcolor=color)

fig.tight_layout()  # otherwise the right y-label is slightly clipped
plt.show()

_______
## 3) A posteriori error estimation

In the following, we try to illustrate the idea of a posteriori error estimation, as for the 2D case, it is a simple poisson equation, we show the residual type error estimate, which can be used to for the adaptive mesh refinement

TBD : 

for now, I just make several refined meshes, and compute their magnetic energy, as well as residual type error estimate
the convergence of the estimator is clear


[to be done]: may be better to show a mesh which is well refined at the corner with less DoFs.. 

voir avec Theodore
et refair le calcul to voir l'estimateur sont "un peu pres le meme grandeur", ainsi l'energie magnetique

In [ ]:
from ngsolve import HCurl, Norm, VOL

def ZZlike_estimate(sol):
    """ recovery-based (Zienkiewicz-Zhu-type) estimator """
    # measure the jump of tangiential components of h
    # heuristic and no garantee... But simpler!
    mesh = sol.space.mesh
    mu = mesh.MaterialCF({"iron" : mu0*mu_r}, default = mu0)
    H_ref = curl(sol) / mu    
    H_hcurl = GridFunction(HCurl(mesh, order = sol.space.globalorder ))
    H_hcurl.Set(H_ref)
    return (H_ref - H_hcurl)**2 * mu

ZZestimateList = [ZZlike_estimate(sol) for sol in solList]
maxZZestimateList = [max(Integrate(err, sol.space.mesh, VOL, element_wise=True)) for err, sol in zip(ZZestimateList, solList)]

In [ ]:
index = 2
Draw(ZZestimateList[index], solList[index].space.mesh)

TODO : Some comments to introduce the error estimate

In [ ]:
from ngsolve import specialcf, L2

def BRV_estimate(sol):
    """ 
    Residual-based a posteriori error estimator (Babuška–Rheinboldt / Verfürth type)
    for 2D magnetostatics in the vector potential formulation.

    Model problem:   curl( nu * curl A ) = J      (nu = 1/mu = reluctivity)

    For every triangle K it computes the indicator

        eta_K^2 =  (h_K^2 / nu)  * || J ||^2_K                      <- element residual
                 + sum_{F in dK} (h_F / (2 nu_F)) * || [H x n] ||^2_F   <- facet jumps

    The global estimate is  eta = sqrt( sum_K eta_K^2 ).
    Reliable (proven upper bound, up to constants) but a bit tedious to implement.

    Returns: a piecewise-constant GridFunction holding eta_K^2 for each element.
    """
    mesh = sol.space.mesh

    # --- Material and fields -------------------------------------------------
    nu_cf = 1/mesh.MaterialCF({"iron" : mu0*mu_r}, default = mu0)
    B = curl(sol)
    H = B * nu_cf

    # --- Geometric quantities -------------------------------------------------
    n = specialcf.normal(2)        # unit normal on element facets (edges in 2D)
    h_elem = specialcf.mesh_size   # local element size h_K
    # Facet size h_F or  element size, which is equivalent in 2D up to a constant factor.
    h_facet = specialcf.netgen_facetsize if hasattr(specialcf, "netgen_facetsize") else specialcf.mesh_size

    # --- Jump of the tangential component of H --------------------------------
    # Physics: the tangential component of H must be continuous across an
    # interface (if there is no surface current). A discrete solution only
    # satisfies this approximately, so the jump measures the error.
    # In 2D the tangential component is  H . t = H_x*n_y - H_y*n_x  (t = n rotated by 90°).
    # H.Other() is the value of H seen from the neighbouring element on the facet.
    H_other = H.Other()
    n_tangent_jump = (H[0] * n[1] - H[1] * n[0]) - (H_other[0] * n[1] - H_other[1] * n[0])

    # --- Space for the indicators ---------------------------------------------
    # One number per element: L2 of order 0 = piecewise constants.
    # Testing a linear form with v_l2 (=1 on K, 0 elsewhere) gives the
    # integral over K, i.e. exactly the local contribution eta_K^2.
    fes_L2 = L2(mesh, order=0)
    u_l2, v_l2 = fes_L2.TnT()

    lf = LinearForm(fes_L2)

    # --- Term 1: element residual ---------------------------------------------
    # Interior residual is  J - curl(H). Here only the data term J is kept,
    # because for low-order elements curl(H) = 0 inside each triangle.
    # With higher-order elements we should add the curl(H) contribution.
    # Only integrated on the coil regions ("condP|condN" = positive/negative
    # conductors), since J = 0 elsewhere.
    # Weight h_K^2 / nu = h_K^2 * mu is consistent with the energy norm.
    lf += (h_elem ** 2)/nu_cf * (J ** 2) * v_l2 * dx("condP|condN")
    
    # --- Term 2: facet jump residual -------------------------------------------
    # Average reluctivity on the two sides of the facet, so the weight
    # stays meaningful at iron/air interfaces where nu jumps strongly.
    nu_avg = 0.5 * (nu_cf + nu_cf.Other())
    # dx(skeleton=True) integrates over all interior facets, each visited once.
    # Testing with (v_l2 + v_l2.Other()) adds the contribution to BOTH neighbouring
    # elements, and the factor 0.5 splits the facet term equally between them.
    lf += 0.5 * h_facet / nu_avg * (n_tangent_jump ** 2) * (v_l2 + v_l2.Other()) * dx(skeleton=True)

    # Assemble: lf.vec[i] now holds eta_K^2 for element i.
    lf.Assemble()

    # --- Output ----------------------------------------------------------------
    # Copy the vector into a GridFunction so it can be drawn and evaluated.
    # Values are eta_K^2. Use eta.vec.FV().NumPy() to get a NumPy array, then
    #   global estimate = sqrt(sum(eta_K^2)),   max indicator = max(eta_K^2)
    # for adaptive refinement (marking elements with the largest eta_K^2).
    eta = GridFunction(fes_L2)
    eta.vec.data = lf.vec

    return eta

BRVestimateList = [BRV_estimate(sol) for sol in solList]
maxBRVestimateList = [max(Integrate(err, sol.space.mesh, VOL, element_wise=True)) for err, sol in zip(BRVestimateList, solList)]

In [ ]:
index = 2
Draw(BRVestimateList[index], solList[index].space.mesh)

### Adaptive mesh refinement

Dolfer marking : 
SOLVE→ESTIMATE→MARK→REFINE→SOLVE→...

we start with a very coarse mesh.

In [ ]:
mesh = gapedInductorrefine(airgap = 0, h = 1)
Draw(mesh)

Then we solve the problem and estimate the local error...

In [ ]:
estimate = BRV_estimate # ZZlike_estimate
sol = solve_magnetostatic(mesh)
err = estimate(sol)
Draw(err, sol.space.mesh)

We mark the elements to refine... (Lets say the element with at least 15% of the max error)

In [ ]:
import numpy as np

proportion = 0.25

eta2 = Integrate(err, mesh, VOL, element_wise=True)
maxerr_adapt = [max(eta2)]
toterr_adapt = [np.sum(eta2)]
n_elts = [mesh.ne]
mesh.ngmesh.Elements2D().NumPy()["refine"] = eta2.NumPy() > proportion*maxerr_adapt[-1]

mesh.Refine()
Draw(mesh)

And we loop!

In [ ]:
for i in range(8):
    sol = solve_magnetostatic(mesh)
    err = estimate(sol)
    eta2 = Integrate(err, mesh, VOL, element_wise=True)
    maxerr_adapt.append(max(eta2))
    toterr_adapt.append(np.sum(eta2))
    n_elts.append(mesh.ne)
    mesh.ngmesh.Elements2D().NumPy()["refine"] = eta2.NumPy() > proportion*maxerr_adapt[-1]
    mesh.Refine()

In [ ]:
Draw(curl(sol), sol.space.mesh)

In [ ]:
# display
n_elts_adapt = np.array(n_elts)
n_elts_uniform = np.array([sol.space.mesh.ne for sol in solList])
err_uniform = np.array(maxBRVestimateList) ** 0.5
err_adaptative = np.array(maxerr_adapt) **0.5
plt.loglog(n_elts_uniform, err_uniform, ':o', label = "uniform refinement")
plt.loglog(n_elts_adapt, err_adaptative, 'o:', label = "adaptive refinement")
plt.loglog(n_elts_uniform, err_uniform[0]/(n_elts_uniform/n_elts_uniform[0]), '--', label = "n")
plt.loglog(n_elts_adapt, err_adaptative[0]/(n_elts_adapt/n_elts_adapt[0])**2, '--', label = "$n^2$")

plt.legend(); plt.grid(); plt.xlabel("h");  plt.ylabel("estimated max(error)")
plt.show()